<a href="https://colab.research.google.com/github/yajima-yasutoshi/Model2026/blob/main/20260805/%E7%A2%BA%E8%AA%8D%E3%83%86%E3%82%B9%E3%83%88%EF%BC%888%E6%9C%885%E6%97%A5%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#準備

In [ ]:
%%capture
!pip install mip

#混合整数問題の最適化

##問題1

以下の混合整数計画問題を解き、最適解を求めよ。

$$
\begin{array}{lll}
\text{最大化} & 5x_1 + 4x_2 + 8x_3 + 6x_4 + 40x_5\\
\text{制約条件}& 2x_1 + 3x_2 + x_3 + 2x_4 + 10x_5 & \le 50\\
& x_1 + 2x_2 + 4x_3 + x_4 + 5 x_5 & \le 45 \\
& 3x_1 + x_2 + 2x_3 + 3x_4 + 8x_5 & \ge 30
\end{array}
$$

ただし、$x_1,x_2$ は非負の連続変数、$x_3,x_4$ は非負の整数変数、
$x_5$ はバイナリ変数とする。

---
---
---
# シフトスケジューリング問題

最終課題としてシフトスケジューリング問題に取り組む。
シフトスケジューリングは、人員配置を最適化し、
コスト削減や従業員の満足度向上に貢献する重要な課題である。

## 問題の概要
シフトスケジューリング問題とは、特定の期間（例：1日、1週間）において、各勤務シフトに必要な人数のスタッフを割り当てる問題である。
この際、スタッフのスキル、労働契約、個人の希望、人件費など、様々な制約条件や目的を考慮に入れる必要がある。

実社会での応用例としては、
看護師や医師の24時間体制のシフト作成がある。
患者のケアに必要なスキルを持つ適切な人数の医療スタッフを各シフトに配置する際に用いられる。

人員不足や過剰な人員配置を避け、法的規制（労働時間の上限、休憩時間など）を遵守しつつ、運用コストの最小化や従業員の公平性・満足度の最大化などを目指す。


## 定式化
ある施設（例：病院の病棟など）における1日の看護師のシフトスケジューリングを考える。

### 問題設定の例

* **期間:** 1日
* **シフト:** 1日は3つのシフトに分割される。
    * 日勤 (Morning)
    * 夕勤 (Evening)
    * 夜勤 (Night)
* **看護師:** 看護師が複数名いる。
* **需要:** 各シフトには、最低限必要な看護師の人数（需要）が定められている。
* **制約:**
    1.  各看護師は、1日に2つのシフトまでしか担当できない。
    2.  各看護師は、特定のシフトに勤務可能かどうかの情報（アベイラビリティ）がある。
* **目的:** 全てのシフトの需要を満たしつつ、勤務する看護師の総数を最小限にする。



### 決定変数

この問題を数理モデルとして表現するために、以下の決定変数を定義する。

* $x_{ij}$: 看護師 $i$ がシフト $j$ に割り当てられる場合に1、そうでない場合に0をとるバイナリ変数。
    * $i \in I = \{1, 2, \dots, N\}$ （看護師の集合）
    * $j \in J = \{\text{Morning, Evening, Night}\}$ （シフトの集合）
* $y_i$: 看護師 $i$ がいずれかのシフトに割り当てられる場合に1、そうでない場合に0をとるバイナリ変数。

### 目的関数

目的は、勤務する看護師の総数の最小化を考える。
これは、変数 $y_i$ を用いて以下のように表される。

$$\text{Minimize} \quad Z = \sum_{i \in I} y_i$$

### 制約条件

以下の制約条件を満たす必要がある。

1.  **シフト充足制約 (Shift Coverage):** 各シフト $j$ には、最低限必要な人数 $D_j$ の看護師が割り当てられなければならない。
$$\sum_{i \in I} x_{ij} \ge D_j \quad \forall j \in J$$

2.  **看護師の最大勤務シフト数制約 (Nurse Assignment Limit):** 各看護師 $i$ は最大2つのシフトまで割り当てが可能なので
$$\sum_{j \in J} x_{ij} \le 2 \quad \forall i \in I$$
となる。

3.  **勤務変数連携制約 (Linking Variables):** 看護師 $i$ がいずれかのシフト $j$ に割り当てられた場合 ($x_{ij}=1$)、その看護師は勤務するもの ($y_i=1$) とする。
$$x_{ij} \le y_i \quad \forall i \in I, \forall j \in J$$
この制約と目的関数（$y_i$ の総和を最小化）により、$x_{ij}$ のいずれかが1になれば $y_i$ は1になり、全ての $x_{ij}$ が0であれば $y_i$ は0になりなる。

4.  **勤務可否制約 (Availability):** 看護師 $i$ がシフト $j$ に勤務できない場合、対応する $x_{ij}$ は0でなければならない。
これは、モデル構築時に、勤務可能な $(i,j)$ のペアに対してのみ変数を生成することで対応する。

5.  **変数型制約 (Variable Type):**
$$x_{ij} \in \{0, 1\} \quad \forall i \in I, \forall j \in J$$
$$y_i \in \{0, 1\} \quad \forall i \in I$$


## 数理モデル

上記をまとめると、シフトスケジューリング問題の数理モデルは以下のように記述される。

**パラメータ:**

* $I$: 看護師の集合
* $J$: シフトの集合
* $D_j$: シフト $j$ に必要な最低看護師数
* $A_{ij}$: 看護師 $i$ がシフト $j$ で勤務可能な場合に1、そうでない場合に0（モデル構築時に考慮）

**決定変数:**

* $x_{ij} \in \{0,1\}$: 看護師 $i$ がシフト $j$ に割り当てられるか否か
* $y_i \in \{0,1\}$: 看護師 $i$ が勤務するか否か

**目的関数:**

$$\text{Minimize} \quad \sum_{i \in I} y_i$$

**制約条件:**

$$\sum_{i \in I \text{ s.t. } A_{ij}=1} x_{ij} \ge D_j \quad \forall j \in J \quad \text{(各シフトで必要な最低人数)}$$
$$\sum_{j \in J \text{ s.t. } A_{ij}=1} x_{ij} \le 2 \quad \forall i \in I \quad \text{(各看護師は多くても2つのシフト)}$$
$$x_{ij} \le y_i \quad \forall i \in I, j \in J \text{ s.t. } A_{ij}=1 \quad \text{(いずれかのシフトに割当られると勤務となる)}$$
$$x_{ij} \in \{0, 1\} \quad \forall i \in I, j \in J \text{ s.t. } A_{ij}=1$$
$$y_i \in \{0, 1\} \quad \forall i \in I$$


## Python MIP を用いた最適化

それでは、具体的なデータを用いて `python-mip` でこの問題を最適化する。

### パラメータ（定数の）設定

例題として、以下のようにパラメータを設定する。

In [ ]:
import pandas as pd

# 問題データ定義
nurses = ["N1", "N2", "N3", "N4", "N5", "N6", "N7", "N8"]
shifts = ["Morning", "Evening", "Night"]

# 各シフトに必要な看護師の数
demand = {"Morning": 3, "Evening": 2, "Night": 1}

# 看護師のアベイラビリティ (タプル (看護師, シフト) をキーとする辞書)
# 値が1なら勤務可能
# 看護師ごとに勤務できる時間帯が異なる
availability_raw = {
    ("N1", "Morning"): 1, ("N1", "Evening"): 1,
    ("N2", "Morning"): 1, ("N2", "Night"): 1,
    ("N3", "Evening"): 1, ("N3", "Night"): 1,
    ("N4", "Morning"): 1, ("N4", "Evening"): 1, ("N4", "Night"): 1,
    ("N5", "Morning"): 1,
    ("N6", "Morning"): 1, ("N6", "Evening"): 1,
    ("N7", "Evening"): 1, ("N7", "Night"): 1,
    ("N8", "Morning"): 1, ("N8", "Night"): 1
}

# 実際に変数を定義する対象となる、勤務可能な (看護師, シフト) のペアのリスト
# get() を使い、キーが存在しない場合には 0 が返されるようにする
available_assignments = [(n, s) for n in nurses for s in shifts if availability_raw.get((n, s), 0) == 1]


看護師と可能なシフトの様子を可視化した。

In [ ]:
#@title パラメータの可視化
# --- ピボットテーブル作成のためのデータ整形 ---
availability_data_for_pivot = []
for nurse in nurses:
    for shift in shifts:
        # availability_raw にキーが存在し、かつ値が1なら勤務可能 (1), それ以外は勤務不可能 (0)
        is_available = availability_raw.get((nurse, shift), 0)
        availability_data_for_pivot.append({
            "看護師": nurse,
            "シフト": shift,
            "勤務可否フラグ": is_available
        })

# Pandas DataFrameに変換
df_availability = pd.DataFrame(availability_data_for_pivot)

# 看護師を行、シフトを列、勤務可否フラグを値とする
pivot_availability = df_availability.pivot_table(
    index="看護師",
    columns="シフト",
    values="勤務可否フラグ",
    fill_value=0 # データが存在しない組み合わせは0（勤務不可能）で埋める
)

# シフトの表示順を固定したい場合 (Morning, Evening, Night の順)
pivot_availability = pivot_availability.reindex(columns=shifts)

# 値を 'O' (勤務可) と 'X' (勤務不可) に変換して見やすくする
pivot_availability_display = pivot_availability.map(lambda x: 'O' if x == 1 else 'X')

# 結果の表示
print("看護師のアベイラビリティ:")
print(pivot_availability_display)

# (オプション) 元のフラグ値のピボットテーブルも表示する場合
# print("\n看護師のアベイラビリティ (ピボットテーブル - フラグ値):")
# print(pivot_availability)

print("\n各シフトに必要な看護師の数")
print(demand)

看護師のアベイラビリティ:
シフト Morning Evening Night
看護師                      
N1        O       O     X
N2        O       X     O
N3        X       O     O
N4        O       O     O
N5        O       X     X
N6        O       O     X
N7        X       O     O
N8        O       X     O

各シフトに必要な看護師の数
{'Morning': 3, 'Evening': 2, 'Night': 1}


---
##問題2

全てのシフトの需要を満たしつつ、勤務する看護師の総数を最小にしたい。
最小の看護師数を求めよ。

---
##問題3

目的を「勤務看護師総数の最小化」から「総人件費の最小化」に変更する。
各看護師が各シフトで働く際の人件費（コスト）は以下のように定義されるとする。
なお、看護師がそのシフトで勤務可能でない場合には、コストの考慮は不要。

In [ ]:
# 人件費データ C_ij
costs = {
    ("N1", "Morning"): 10, ("N1", "Evening"): 12,
    ("N2", "Morning"): 11, ("N2", "Night"): 14,
    ("N3", "Evening"): 11, ("N3", "Night"): 13,
    ("N4", "Morning"): 10, ("N4", "Evening"): 12, ("N4", "Night"): 15,
    ("N5", "Morning"): 8,
    ("N6", "Morning"): 18, ("N6", "Evening"): 9,
    ("N7", "Evening"): 5, ("N7", "Night"): 10,
    ("N8", "Morning"): 8, ("N8", "Night"): 11,

}

---
##問題4
問題3に、
看護師 N6 と看護師 N7 が同じシフトに同時に割り当てられないようにする制約を追加し、
総人件費の最小化を行え（例：N6が夜勤なら、N7は夜勤であってはならない。）


---
##問題5

以下のスキル要件を導入する。
* **スキル:** 「リーダスキル (Lead)」と「通常スキル (Regular)」の2種類がある。
* **看護師のスキル:**

In [ ]:
# スキルデータ
skills = ["Lead", "Regular"]
nurse_skills = {
    "N1": "Lead",
    "N2": "Regular",
    "N3": "Regular",
    "N4": "Lead",
    "N5": "Regular",
    "N6": "Lead",
    "N7": "Regular",
    "N8": "Regular",

}

* **シフトのスキル要件:**
    * 各シフトで最低1名の「Lead」スキルを持つ看護師が必要。

**問題3**
の目的と制約（需要充足、最大2シフト/人、アベイラビリティ）の場合で、
上のスキル要件を考慮し、総人件費を最小化する。

**注意**
問題4で設定した、「看護師 N6 と看護師 N7 が、同じシフトに同時に割り当てられないようにする制約」は考えないことに注意せよ。